# Thực nghiệm ABSA – Clean Ablation Training

Notebook này phục vụ huấn luyện và đánh giá các biến thể mô hình cho bài toán Aspect-Based Sentiment Analysis (ABSA) trên tập dữ liệu restaurant.

Mục tiêu của notebook:

1. Khởi tạo một phiên thực nghiệm có thể tái lập trên Kaggle.
2. Huấn luyện lại toàn bộ mô hình từ đầu, không sử dụng checkpoint hoặc log huấn luyện có sẵn.
3. So sánh các cấu hình ablation chính: `CLS`, `aspect_attention`, `hybrid pooling`, và `neutral class weighting`.
4. Xuất toàn bộ kết quả tuning/final evaluation thành file `.zip` để phân tích sau huấn luyện.

Yêu cầu khi chạy trên Kaggle:

- Bật **GPU**.
- Bật **Internet** nếu cần tải pretrained model lần đầu.
- Thêm file `absa_clean_ablation_fresh.zip` vào phần **Input Dataset** của notebook.


In [1]:
from pathlib import Path
import os, shutil, zipfile, json, subprocess, sys

print("Python:", sys.version)
print("Kaggle working exists:", Path("/kaggle/working").exists())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found; GPU may be disabled.")


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle working exists: True
Sat Jun 20 15:47:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |          

## 1. Khởi tạo môi trường thực nghiệm và nạp mã nguồn

Bước này thiết lập thư mục làm việc, chuẩn hóa trạng thái phiên chạy và giải nén package mã nguồn từ Kaggle Input vào `/kaggle/working`.

Việc chuẩn hóa trạng thái giúp đảm bảo kết quả huấn luyện được sinh ra từ lần chạy hiện tại, không bị ảnh hưởng bởi artifact được tạo từ các lần chạy trước trong cùng phiên Kaggle.


In [2]:
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT = Path('/kaggle/input/datasets/banhbaogachcua/nlp-melaycon')
PROJECT = WORK / 'absa_clean_ablation_fresh'

CLEAN_RUN = True
if CLEAN_RUN:
    for path in [
        PROJECT,
        WORK / 'tuning_ablation_core',
        WORK / 'outputs_ablation_core',
        WORK / 'tuning_ablation_desc',
        WORK / 'outputs_ablation_desc',
        WORK / 'absa_ablation_results_fresh.zip',
    ]:
        if path.exists():
            print('Resetting generated artifact path:', path)
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()

zip_candidates = []
if INPUT.exists():
    zip_candidates = list(INPUT.rglob('absa_clean_ablation_fresh.zip'))
    if not zip_candidates:
        zip_candidates = list(INPUT.rglob('absa_clean_ablation*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK)
else:
    train_candidates = list(INPUT.rglob('train.py')) if INPUT.exists() else []
    if not train_candidates:
        raise FileNotFoundError('Không tìm thấy absa_clean_ablation_fresh.zip hoặc train.py trong /kaggle/input')
    src = train_candidates[0].parent
    print('Copying source folder:', src)
    shutil.copytree(src, PROJECT, dirs_exist_ok=True)

if not (PROJECT / 'train.py').exists():
    candidates = [p.parent for p in WORK.rglob('train.py') if 'absa_clean_ablation' in str(p.parent)]
    if not candidates:
        raise FileNotFoundError('Extract xong nhưng không tìm thấy train.py')
    PROJECT = candidates[0]

print('PROJECT =', PROJECT)
print('Files:', sorted([p.name for p in PROJECT.iterdir()])[:30])


Copying source folder: /kaggle/input/datasets/banhbaogachcua/nlp-melaycon/absa_clean_ablation_fresh
PROJECT = /kaggle/working/absa_clean_ablation_fresh
Files: ['CLEAN_START.md', 'KAGGLE_FRESH.md', 'PATCH_NOTES.md', 'README.md', 'absa_pipeline.py', 'dataset', 'docs', 'kaggle_ablation_fresh.ipynb', 'requirements.txt', 'run_ablation_core.sh', 'run_ablation_desc.sh', 'train.py', 'train_restaurant.sh', 'tune.py']


## 2. Cài đặt thư viện phụ thuộc

Bước này cài đặt các thư viện cần thiết được khai báo trong `requirements.txt`. Sau bước này, notebook có thể gọi trực tiếp các script huấn luyện và tuning của project.


In [3]:
os.chdir(PROJECT)
print('cwd =', Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


cwd = /kaggle/working/absa_clean_ablation_fresh
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 3. Xác minh trạng thái clean-start của package

Bước này kiểm tra package sau khi giải nén để xác nhận rằng mã nguồn không chứa checkpoint, thư mục kết quả, cache huấn luyện hoặc artifact lịch sử.

Đây là bước kiểm soát tính tái lập trước khi bắt đầu huấn luyện.


In [4]:
os.chdir(PROJECT)
forbidden_markers = ['tuning_', 'outputs_', 'wandb', 'runs', 'logs', '__pycache__']
forbidden_suffixes = {'.pt', '.pth', '.bin', '.safetensors'}
found = []
for p in PROJECT.rglob('*'):
    rel = str(p.relative_to(PROJECT))
    if any(part in rel for part in forbidden_markers) or p.suffix in forbidden_suffixes:
        found.append(rel)

print('Generated/history-like artifacts detected:', len(found))
for item in found[:50]:
    print('-', item)
assert not found, 'Clean-start validation failed: package contains output, checkpoint, cache, or history-like artifacts.'

print('\nDataset files:')
for p in sorted(Path('dataset/rest').glob('*')):
    print('-', p, p.stat().st_size)

print('\nSearch-space help:')
subprocess.run([sys.executable, 'tune.py', '--help'], check=False)


Generated/history-like artifacts detected: 0

Dataset files:
- dataset/rest/aspects.json 444731
- dataset/rest/test.multiple.json 1202702
- dataset/rest/train.multiple.json 4078526

Search-space help:
usage: tune.py [-h] [--dataset {lap,rest,twi}] [--model MODEL]
               [--epochs EPOCHS] [--final-epochs FINAL_EPOCHS]
               [--patience PATIENCE] [--batch-size BATCH_SIZE]
               [--gradient-accumulation-steps GRADIENT_ACCUMULATION_STEPS]
               [--weight-decay WEIGHT_DECAY] [--valid-ratio VALID_RATIO]
               [--split-seed SPLIT_SEED] [--tuning-seeds TUNING_SEEDS]
               [--validation-seed-pairs VALIDATION_SEED_PAIRS]
               [--final-seeds FINAL_SEEDS] [--output-dir OUTPUT_DIR]
               [--final-output-dir FINAL_OUTPUT_DIR] [--amp]
               [--gradient-checkpointing] [--run-final]
               [--keep-trial-checkpoints] [--resume]
               [--search-space {ablation_all,ablation_core,ablation_desc,initial,refined}

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--help'], returncode=0)

## 4. Huấn luyện core ablation từ đầu

Bước này chạy nhóm thí nghiệm chính (`ablation_core`) nhằm đánh giá riêng từng thay đổi quan trọng trong mô hình. Nhóm này **không sử dụng aspect description** để tránh gộp nhiều yếu tố trong cùng một trial.

Các trial chính:

- `baseline_cls`: cấu hình đối chứng dùng CLS pooling.
- `aspect_attention_term`: dùng aspect-aware attention trên input term.
- `hybrid_term`: kết hợp CLS representation và aspect-aware representation.
- `cls_neutral125`: tăng trọng số lớp neutral ở mức 1.25.
- `cls_neutral150`: tăng trọng số lớp neutral ở mức 1.50.
- `aspect_attention_neutral125`: kết hợp aspect-aware attention và neutral weighting mức 1.25.

Nếu GPU bị thiếu bộ nhớ, có thể giảm `BATCH_SIZE` xuống 4 và tăng `GRAD_ACCUM` lên 4 để giữ effective batch size gần tương đương.


In [5]:
os.chdir(PROJECT)

VALID_RATIO = 0.15
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 12
FINAL_EPOCHS = 15
SEED_PAIRS = '2024:2024,42:42,3407:3407'
FINAL_SEEDS = '2024,42,3407'

cmd = [
    sys.executable, 'tune.py',
    '--dataset', 'rest',
    '--search-space', 'ablation_core',
    '--validation-seed-pairs', SEED_PAIRS,
    '--valid-ratio', str(VALID_RATIO),
    '--final-seeds', FINAL_SEEDS,
    '--epochs', str(EPOCHS),
    '--final-epochs', str(FINAL_EPOCHS),
    '--patience', '3',
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--amp',
    '--gradient-checkpointing',
    '--run-final',
    '--output-dir', str(WORK / 'tuning_ablation_core'),
    '--final-output-dir', str(WORK / 'outputs_ablation_core'),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


/usr/bin/python3 tune.py --dataset rest --search-space ablation_core --validation-seed-pairs 2024:2024,42:42,3407:3407 --valid-ratio 0.15 --final-seeds 2024,42,3407 --epochs 12 --final-epochs 15 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --amp --gradient-checkpointing --run-final --output-dir /kaggle/working/tuning_ablation_core --final-output-dir /kaggle/working/outputs_ablation_core
RUN: /usr/bin/python3 train.py --dataset rest --model microsoft/deberta-v3-base --epochs 12 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --weight-decay 0.01 --valid-ratio 0.15 --split-seed 2024 --seed 2024 --output-dir /kaggle/working/tuning_ablation_core --run-name tune_baseline_cls_split2024_seed2024 --evaluation-split validation --pooling-strategy cls --input-format term --encoder-layer-pooling last --encoder-learning-rate 8e-06 --head-learning-rate 2e-05 --dropout 0.3 --class-weighting none --neutral-weight-multiplier 1.0 --label-smoothing 0.02 --loss-type cross_entropy

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-20 15:48:09.736778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781970489.958742      63 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781970490.017481      63 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3070,
      "labels": {
        "negative": 686,
        "neutral": 542,
        "positive": 1842
      },
      "unique_aspects": 1069
    },
    "train_all": {
      "samples": 3607,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2163
      },
      "unique_aspects": 1201,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.038
    },
    "valid": {
      "samples": 537,
      "labels": {
        "negative": 121,
        "neutral": 95,
        "positive": 321
      },
      "unique_aspects": 281,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.2551
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 520,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-20 16:02:06.238612: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781971326.256388     107 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781971326.262567     107 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3066,
      "labels": {
        "negative": 684,
        "neutral": 540,
        "positive": 1842
      },
      "unique_aspects": 1066
    },
    "train_all": {
      "samples": 3607,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2163
      },
      "unique_aspects": 1201,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.038
    },
    "valid": {
      "samples": 541,
      "labels": {
        "negative": 123,
        "neutral": 97,
        "positive": 321
      },
      "unique_aspects": 273,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.2532
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 520,
      "unseen_aspect_samples": 364,
      "unseen_aspect_ratio": 0.325
    },
   

...

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-20 20:36:12.886501: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781987772.904329     594 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781987772.910525     594 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3070,
      "labels": {
        "negative": 686,
        "neutral": 542,
        "positive": 1842
      },
      "unique_aspects": 1069
    },
    "train_all": {
      "samples": 3607,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2163
      },
      "unique_aspects": 1201,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.038
    },
    "valid": {
      "samples": 537,
      "labels": {
        "negative": 121,
        "neutral": 95,
        "positive": 321
      },
      "unique_aspects": 281,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.2551
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 520,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-20 20:52:12.121546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781988732.141286     622 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781988732.147669     622 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3070,
      "labels": {
        "negative": 686,
        "neutral": 542,
        "positive": 1842
      },
      "unique_aspects": 1069
    },
    "train_all": {
      "samples": 3607,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2163
      },
      "unique_aspects": 1201,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.038
    },
    "valid": {
      "samples": 537,
      "labels": {
        "negative": 121,
        "neutral": 95,
        "positive": 321
      },
      "unique_aspects": 281,
      "unseen_aspect_samples": 137,
      "unseen_aspect_ratio": 0.2551
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 520,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--dataset', 'rest', '--search-space', 'ablation_core', '--validation-seed-pairs', '2024:2024,42:42,3407:3407', '--valid-ratio', '0.15', '--final-seeds', '2024,42,3407', '--epochs', '12', '--final-epochs', '15', '--patience', '3', '--batch-size', '8', '--gradient-accumulation-steps', '2', '--amp', '--gradient-checkpointing', '--run-final', '--output-dir', '/kaggle/working/tuning_ablation_core', '--final-output-dir', '/kaggle/working/outputs_ablation_core'], returncode=0)

## 5. Tổng hợp nhanh kết quả core ablation

Bước này đọc các file kết quả được sinh ra sau tuning và final evaluation, sau đó hiển thị các metric chính để so sánh giữa các trial.

Các metric cần chú ý gồm `Macro-F1`, độ lệch chuẩn giữa các seed, `Robust Macro-F1`, per-seed final metrics và calibration error nếu có.


In [6]:
import pandas as pd

summary_path = WORK / 'tuning_ablation_core' / 'tuning_summary.csv'
final_path = WORK / 'outputs_ablation_core' / 'final_runs_summary.csv'
meanstd_path = WORK / 'outputs_ablation_core' / 'final_metrics_mean_std.json'

print('Tuning summary:', summary_path)
summary = pd.read_csv(summary_path)
cols = [c for c in ['trial','mean_macro_f1','std_macro_f1','robust_macro_f1','mean_accuracy','mean_ece','pooling_strategy','input_format','neutral_weight_multiplier'] if c in summary.columns]
display(summary[cols])

if final_path.exists():
    print('Final runs:', final_path)
    final = pd.read_csv(final_path)
    display(final)

if meanstd_path.exists():
    print(json.dumps(json.loads(meanstd_path.read_text()), indent=2))


Tuning summary: /kaggle/working/tuning_ablation_core/tuning_summary.csv


,trial,mean_macro_f1,std_macro_f1,robust_macro_f1,mean_accuracy,mean_ece,pooling_strategy,input_format,neutral_weight_multiplier
0,aspect_attention_neutral125,0.773800,0.007605,0.766195,0.833067,0.039733,aspect_attention,term,1.25
1,aspect_attention_term,0.763367,0.011146,0.752220,0.825600,0.035100,aspect_attention,term,1.00
2,hybrid_term,0.758300,0.006556,0.751744,0.821900,0.048533,hybrid,term,1.00
3,cls_neutral125,0.753767,0.030605,0.723161,0.818200,0.042167,cls,term,1.25
4,cls_neutral150,0.753200,0.031325,0.721875,0.816367,0.044700,cls,term,1.50
5,baseline_cls,0.753100,0.036736,0.716364,0.818833,0.048833,cls,term,1.00


Final runs: /kaggle/working/outputs_ablation_core/final_runs_summary.csv


,seed,best_epoch,accuracy,precision_macro,recall_macro,macro_f1,weighted_f1,roc_auc_macro,log_loss,expected_calibration_error,temperature
0,2024,12,0.8652,0.8275,0.7656,0.7810,0.8520,0.9111,0.4201,0.0459,1.883212
1,42,4,0.8732,0.8285,0.7821,0.7995,0.8652,0.9573,0.3355,0.0344,1.560496
2,3407,3,0.8571,0.8116,0.7503,0.7706,0.8443,0.9457,0.3707,0.0439,1.125398


{
  "accuracy": {
    "mean": 0.865167,
    "std": 0.006573
  },
  "precision_macro": {
    "mean": 0.822533,
    "std": 0.007742
  },
  "recall_macro": {
    "mean": 0.766,
    "std": 0.012985
  },
  "macro_f1": {
    "mean": 0.7837,
    "std": 0.011952
  },
  "weighted_f1": {
    "mean": 0.853833,
    "std": 0.00863
  },
  "roc_auc_macro": {
    "mean": 0.938033,
    "std": 0.019625
  },
  "log_loss": {
    "mean": 0.375433,
    "std": 0.0347
  },
  "expected_calibration_error": {
    "mean": 0.0414,
    "std": 0.005017
  }
}


## 6. Thực nghiệm bổ sung: aspect description ablation

Bước này là thí nghiệm tùy chọn để kiểm tra riêng tác động của `aspects.json`. Mặc định cell không chạy để ưu tiên nhóm core ablation trước.

Chỉ nên bật `RUN_DESC_ABLATION = True` sau khi đã có kết quả core ablation, nhằm tránh nhầm lẫn giữa đóng góp của aspect description với các thay đổi như pooling hoặc class weighting.


In [7]:
RUN_DESC_ABLATION = False  # Set to True only after completing core ablation if description-based trials are required.

if RUN_DESC_ABLATION:
    desc_cmd = [
        sys.executable, 'tune.py',
        '--dataset', 'rest',
        '--search-space', 'ablation_desc',
        '--validation-seed-pairs', SEED_PAIRS,
        '--valid-ratio', str(VALID_RATIO),
        '--final-seeds', FINAL_SEEDS,
        '--epochs', str(EPOCHS),
        '--final-epochs', str(FINAL_EPOCHS),
        '--patience', '3',
        '--batch-size', str(BATCH_SIZE),
        '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--amp',
        '--gradient-checkpointing',
        '--run-final',
        '--output-dir', str(WORK / 'tuning_ablation_desc'),
        '--final-output-dir', str(WORK / 'outputs_ablation_desc'),
    ]
    print(' '.join(desc_cmd))
    subprocess.run(desc_cmd, check=True)
else:
    print('Description ablation is disabled. Set RUN_DESC_ABLATION = True to execute this optional experiment.')


Description ablation is disabled. Set RUN_DESC_ABLATION = True to execute this optional experiment.


## 7. Đóng gói kết quả thực nghiệm

Bước này gom các file kết quả tuning và final evaluation thành một file `.zip` để tải về từ Kaggle.

Mặc định notebook không đưa checkpoint vào file nén để giảm dung lượng. Nếu cần lưu checkpoint phục vụ phân tích sâu hơn, đặt `INCLUDE_CHECKPOINTS = True`.


In [8]:
INCLUDE_CHECKPOINTS = False
zip_out = WORK / 'absa_ablation_results_fresh.zip'
folders = [WORK / 'tuning_ablation_core', WORK / 'outputs_ablation_core', WORK / 'tuning_ablation_desc', WORK / 'outputs_ablation_desc']

with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        if not folder.exists():
            continue
        for path in folder.rglob('*'):
            if path.is_file():
                if not INCLUDE_CHECKPOINTS and path.suffix in {'.pt', '.pth', '.bin', '.safetensors'}:
                    continue
                zf.write(path, path.relative_to(WORK))

print('Created:', zip_out)
print('Size MB:', round(zip_out.stat().st_size / 1024 / 1024, 2))


Created: /kaggle/working/absa_ablation_results_fresh.zip
Size MB: 1600.14
